# Probability Theory and Probabilistic Models

This lab has three parts, each taking a different probabilistic tool to behavioral data.

**Part 1** fits a Poisson process to response timing and then checks whether the model's own
predictions hold. You will estimate a rate by maximum likelihood, put a confidence interval on it,
and run three diagnostics that can reject the model. Two records are supplied that return the same
rate estimate; only the diagnostics tell them apart.

**Part 2** applies Bayesian updating to functional analysis data. Starting from a uniform prior
over four candidate functions, you will update after every session and watch the posterior move.
You will run the procedure on a clear case and on an ambiguous one, which behave very differently.

**Part 3** uses Monte Carlo simulation to put a confidence interval on a behavioral parameter
without an analytic formula.

## Background

A standard functional analysis (FA; Iwata et al., 1982/1994) arranges conditions -- here attention,
escape, tangible, and play (control) -- to identify the maintaining variable for problem behavior.
Clinicians typically rely on visual analysis. A Bayesian approach offers a quantitative complement:
state what each candidate function predicts, then let the data move a probability distribution over
those candidates.

Throughout, notice that a probabilistic model commits to more than a mean. It commits to a whole
distribution, which is what makes it possible for data to contradict it.

## Task 1: Import Libraries

Import `pandas`, `numpy`, `matplotlib.pyplot`, and `scipy.stats`. Set a random seed so your
Monte Carlo results in Part 3 are reproducible.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)

---
# Part 1: The Poisson Process as a Model of Response Timing

`operant_irt_data.csv` holds two 50-minute records, `M-01` and `M-02`. Each row is one response,
identified by the subject and the time in seconds from the start of the session. Everything else
you need you will derive.

## Task 2: Load the Operant Data and Derive Inter-Response Times

Load `operant_irt_data.csv`. For each subject, compute:

- the vector of inter-response times (IRTs), which are the differences between successive response
  times;
- the number of responses in each successive 1-minute bin across the 50-minute session.

Report how many responses each subject emitted and the mean IRT for each. Before going further,
note whether the two records look similar on these summary numbers.

In [ ]:
SESSION_S = 3000.0            # 50-minute session
BIN_S = 60.0                  # 1-minute bins
BIN_EDGES = np.arange(0, SESSION_S + BIN_S, BIN_S)

ops = pd.read_csv("operant_irt_data.csv")
print(ops.head())

records = {}
for subject, g in ops.groupby("subject"):
    times = g["time_s"].sort_values().values
    records[subject] = {
        "times": times,
        "irts": np.diff(times),
        "counts": np.histogram(times, bins=BIN_EDGES)[0],
    }

for subject, rec in records.items():
    print(f"{subject}: {len(rec['times'])} responses, "
          f"mean IRT = {rec['irts'].mean():.2f} s")

## Task 3: Maximum Likelihood Estimate of the Rate

For a Poisson process observed over a fixed duration, the maximum likelihood estimate of the rate
is the total number of events divided by the total observation time:

$$\hat{\lambda} = \frac{N}{T}$$

Compute $\hat{\lambda}$ for each subject in responses per minute. Then compute the standard error
using $SE(\hat{\lambda}) = \sqrt{\hat{\lambda}/n}$, where $n$ is the number of 1-minute bins,
and form an approximate 95% confidence interval.

Do the two subjects differ in rate?

In [ ]:
SESSION_MIN = SESSION_S / 60.0

for subject, rec in records.items():
    n_bins = len(rec["counts"])
    lam = len(rec["times"]) / SESSION_MIN          # responses per minute
    se = np.sqrt(lam / n_bins)
    lo, hi = lam - 1.96 * se, lam + 1.96 * se
    rec.update(lam=lam, se=se, ci=(lo, hi))
    print(f"{subject}: lambda-hat = {lam:.2f}/min, SE = {se:.2f}, "
          f"95% CI = [{lo:.2f}, {hi:.2f}]")

print("\nThe confidence intervals overlap substantially: on rate alone, "
      "these two records are not distinguishable.")

## Task 4: The Variance Check

The Poisson distribution has one parameter, so it makes a prediction it cannot wriggle out of: the
variance of the counts equals their mean. Nothing was fitted to the variance, which is what makes
this a real test.

For each subject, compute the mean and variance of the per-minute counts and their ratio. A ratio
near 1 is consistent with the model. A ratio well above 1 is overdispersion; well below 1 is
underdispersion. Which subject fails this check, and in which direction?

In [ ]:
for subject, rec in records.items():
    counts = rec["counts"]
    mean, var = counts.mean(), counts.var(ddof=1)
    rec["dispersion"] = var / mean
    print(f"{subject}: count mean = {mean:5.2f}, variance = {var:5.2f}, "
          f"variance/mean = {var/mean:.2f}")

print("\nM-01 sits at about 1, as the Poisson model requires. "
      "M-02 is roughly 4x overdispersed: far more variable than a constant-rate "
      "process can produce, even though its rate estimate matched M-01's.")

## Task 5: The Inter-Response Time Distribution

A Poisson process at rate $\lambda$ implies that IRTs follow an exponential distribution:

$$f(\tau) = \lambda e^{-\lambda \tau}$$

For each subject, plot a density histogram of the observed IRTs and overlay the exponential density
implied by that subject's own $\hat{\lambda}$. Work in seconds, so convert your rate from
responses per minute back to responses per second.

Where does each record depart from the prediction, and in which direction?

In [ ]:
tau = np.linspace(0, 24, 400)
edges = np.arange(0, 24.5, 1.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (subject, rec) in zip(axes, records.items()):
    lam_s = rec["lam"] / 60.0                       # responses per second
    ax.hist(rec["irts"], bins=edges, density=True, color="lightgray",
            edgecolor="gray")
    ax.plot(tau, lam_s * np.exp(-lam_s * tau), color="firebrick", lw=2.5,
            label=f"Exponential($\\hat{{\\lambda}}$ = {lam_s:.3f}/s)")
    ax.set_xlabel("Inter-response time (s)")
    ax.set_title(subject)
    ax.legend()
axes[0].set_ylabel("Density")
plt.tight_layout()
plt.show()

print("M-01 tracks the exponential closely. M-02 has far too many very short IRTs "
      "(the within-bout responses) and too many long ones (the pauses), with too "
      "little probability in between. Its mean IRT is still correct, which is why "
      "the rate estimate did not catch the problem.")

## Task 6: Quantile-Quantile Plot

A histogram makes gross departures visible; a Q-Q plot makes the shape of the departure legible.
Plot the observed IRT quantiles against the theoretical quantiles of the exponential distribution
implied by each subject's $\hat{\lambda}$. Add the diagonal. Points on the diagonal indicate
agreement.

The theoretical quantile at probability $p$ for an exponential with rate $\lambda$ is
$-\ln(1 - p)/\lambda$.

Describe the shape of the departure for the subject that fails. What does a curve that sits below
the diagonal at short quantiles and rises above it at long quantiles tell you about the process?

In [ ]:
probs = np.linspace(0.005, 0.995, 200)

fig, ax = plt.subplots(figsize=(6.5, 6))
for (subject, rec), color, marker in zip(records.items(),
                                         ["black", "firebrick"], ["o", "^"]):
    lam_s = rec["lam"] / 60.0
    theoretical = -np.log(1 - probs) / lam_s
    observed = np.quantile(rec["irts"], probs)
    ax.plot(theoretical, observed, ls="none", marker=marker, ms=4,
            color=color, label=subject)

lims = (0, 45)
ax.plot(lims, lims, color="gray", ls="--", lw=1.2)
ax.set_xlim(0, 30); ax.set_ylim(lims)
ax.set_xlabel("Theoretical exponential quantile (s)")
ax.set_ylabel("Observed quantile (s)")
ax.set_title("Q-Q Plot: Observed IRTs vs. the Exponential Prediction")
ax.legend()
plt.tight_layout()
plt.show()

print("M-01 falls on the diagonal. M-02 sits below the diagonal at short quantiles "
      "and climbs steeply above it at long ones: too many very short IRTs and too "
      "many very long ones relative to a single exponential. That S-shape is the "
      "signature of a mixture of two rates rather than one constant rate, which is "
      "exactly how M-02 was generated (bouts of fast responding separated by pauses).")

## Task 7: What the Diagnostics Bought You

Both subjects returned nearly the same $\hat{\lambda}$ with overlapping confidence intervals.
Summarize, in a short printed statement or a markdown cell, what each of the three checks
(variance-to-mean, IRT histogram, Q-Q plot) revealed that the point estimate did not.

Then answer: if you had reported only the mean response rate for these two subjects, what would you
have missed, and would any behavioral conclusion have changed?

In [ ]:
print("Rate estimate alone:")
for subject, rec in records.items():
    print(f"  {subject}: {rec['lam']:.2f}/min, 95% CI "
          f"[{rec['ci'][0]:.2f}, {rec['ci'][1]:.2f}]  -- indistinguishable")

print("\nAfter the diagnostics:")
for subject, rec in records.items():
    verdict = "consistent with a Poisson process" if rec["dispersion"] < 1.5 \
        else "rejects the constant-rate and independence assumptions"
    print(f"  {subject}: variance/mean = {rec['dispersion']:.2f} -- {verdict}")

print(
    "\nReporting only mean rate would have treated these two organisms as behaving "
    "identically. They are not. M-01 responds at a steady rate; M-02 responds in "
    "bouts separated by pauses, which implies a different underlying process and a "
    "different account of what controls the behavior. Any manipulation targeting "
    "bout initiation rather than within-bout rate would be expected to affect the "
    "two records differently, and a rate-only analysis could not detect that."
)

---
# Part 2: Bayesian Updating for Functional Assessment

Two FA datasets are supplied. Both cycle through attention, escape, tangible, and play in a fixed
order, and both carry the session `count`, the `duration_min`, and the resulting `rate_per_min`.

- `functional_analysis_data.csv` -- a clear case, 10-minute sessions.
- `functional_analysis_ambiguous.csv` -- a hard case, 5-minute sessions.

You will build the updating machinery on the clear case, then turn it loose on the ambiguous one.

## Task 8: Load and Inspect Both Datasets

Load both files. For each, report the number of sessions per condition and the mean rate per
condition. Then produce the standard FA graph for each: rate per minute on the y-axis against
session number on the x-axis, one connected series per condition.

For each dataset, write down the conclusion you would reach from visual analysis alone, and how
confident you would be.

In [ ]:
clear_df = pd.read_csv("functional_analysis_data.csv")
ambig_df = pd.read_csv("functional_analysis_ambiguous.csv")

for name, df in [("CLEAR", clear_df), ("AMBIGUOUS", ambig_df)]:
    print(f"--- {name} ({df.duration_min.iloc[0]:.0f}-min sessions) ---")
    print(df.groupby("condition")[["count", "rate_per_min"]]
            .agg(["mean", "std", "count"]).round(2))
    print()

CONDITIONS = ["attention", "escape", "tangible", "play"]
MARKERS = {"attention": "o", "escape": "s", "tangible": "^", "play": "D"}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, (name, df) in zip(axes, [("Clear", clear_df), ("Ambiguous", ambig_df)]):
    for cond in CONDITIONS:
        sub = df[df.condition == cond].sort_values("session")
        ax.plot(sub.session, sub.rate_per_min, marker=MARKERS[cond], label=cond)
    ax.set_xlabel("Session")
    ax.set_ylabel("Rate per minute")
    ax.set_title(f"{name}: Functional Analysis")
    ax.legend(title="Condition", fontsize=8)
plt.tight_layout()
plt.show()

print("Clear: attention is separated from everything else in every cycle. Visual "
      "analysis settles this immediately and with high confidence.\n"
      "Ambiguous: escape is highest on average, but the series overlap and escape "
      "is not highest in every cycle. Visual analysis would be tentative here, and "
      "two analysts could reasonably disagree.")

## Task 9: Set Up the Prior

Define a uniform prior over four candidate functions: `attention`, `escape`, `tangible`, and
`automatic`. Store it as an array or dictionary and plot it as a bar chart.

Note what the fourth hypothesis is doing here. There is no "automatic" *condition* in an FA. What
distinguishes automatic reinforcement is that the behavior persists regardless of the social
consequence arranged, so it predicts elevated responding in *every* condition, including the play
control. Keep that in mind for the next task.

In [ ]:
FUNCTIONS = ["attention", "escape", "tangible", "automatic"]
prior = np.full(len(FUNCTIONS), 1.0 / len(FUNCTIONS))
print("Prior:", {f: float(p) for f, p in zip(FUNCTIONS, prior)})

fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.bar(FUNCTIONS, prior, color="gray")
ax.set_ylim(0, 1)
ax.set_ylabel("P(function)")
ax.set_title("Prior Distribution (uniform)")
plt.tight_layout()
plt.show()

## Task 10: Define the Likelihood

The likelihood is where the behavioral theory enters. Each candidate function makes a claim about
which conditions should evoke elevated responding:

- `attention`, `escape`, and `tangible` predict elevation **only** in their matching condition.
- `automatic` predicts elevation in **every** condition, including play.
- In any condition where a function does not predict elevation, it predicts responding at a base
  rate.

Model the session count as Poisson. If a function predicts elevation for a session's condition, the
expected count is $\lambda_{\text{elevated}} \times \text{duration}$; otherwise it is
$\lambda_{\text{base}} \times \text{duration}$. Then

$$P(\text{count} \mid \text{function}) = \text{Poisson}(\text{count} \mid \lambda \times \text{duration})$$

Write a function returning the **log** likelihood for one session under one hypothesized function.
Work in logs throughout: the products in Bayes' theorem become sums, which is both numerically
safer and exactly the move the chapter makes when it switches to log-likelihood.

For the clear dataset, take $\lambda_{\text{base}} = 1.0$ and $\lambda_{\text{elevated}} = 9.0$
responses per minute. These are modeling assumptions, justified by the condition means you computed
in Task 8. You will test how much they matter in Task 13.

In [ ]:
def log_likelihood(function, condition, count, duration_min,
                   base_rate, elevated_rate):
    """log P(count | function) for a single session, under a Poisson model."""
    # Automatic reinforcement predicts elevation regardless of the social
    # consequence arranged, so it predicts elevation in every condition.
    predicts_elevation = (function == "automatic") or (condition == function)
    lam = elevated_rate if predicts_elevation else base_rate
    return stats.poisson.logpmf(count, lam * duration_min)


# Sanity check: an elevated attention session should favour attention and
# automatic equally, since both predict elevation in the attention condition.
demo = {f: round(float(log_likelihood(f, "attention", 84, 10.0, 1.0, 9.0)), 2)
        for f in FUNCTIONS}
print("log-likelihoods for an 84-response attention session:", demo)

## Task 11: Implement the Updating Loop

Write a function that starts from the uniform prior and updates once per session, in session order,
returning the posterior after every session so you can plot the trajectory.

In log space, Bayes' theorem for each step is

$$\log p(\text{function} \mid \text{data}) = \log p(\text{function}) + \log P(\text{data} \mid \text{function}) - \log P(\text{data})$$

Rather than computing $\log P(\text{data})$ directly, accumulate the unnormalized log posterior
and normalize only when you need probabilities: subtract the maximum, exponentiate, and divide by
the sum. This avoids underflow when one hypothesis becomes overwhelmingly favored.

Run it on the **clear** dataset and print the posterior after each of the first four sessions.

Look carefully at what happens after session 1, and then at what session 2 does. Explain it.

In [ ]:
def normalize(log_p):
    """Turn an unnormalized log posterior into probabilities."""
    shifted = log_p - log_p.max()
    p = np.exp(shifted)
    return p / p.sum()


def run_updating(df, base_rate, elevated_rate):
    """Posterior over FUNCTIONS after each session, starting from a uniform prior."""
    log_post = np.log(np.full(len(FUNCTIONS), 1.0 / len(FUNCTIONS)))
    history = [normalize(log_post)]
    for _, row in df.sort_values("session").iterrows():
        log_post = log_post + np.array([
            log_likelihood(f, row["condition"], row["count"],
                           row["duration_min"], base_rate, elevated_rate)
            for f in FUNCTIONS
        ])
        history.append(normalize(log_post))
    out = pd.DataFrame(history, columns=FUNCTIONS)
    out.insert(0, "session", range(len(history)))
    return out


clear_history = run_updating(clear_df, base_rate=1.0, elevated_rate=9.0)
print(clear_history.head(5).round(4).to_string(index=False))

print(
    "\nAfter session 1 (an elevated attention session) the posterior is split "
    "exactly 0.50/0.50 between attention and automatic. Both hypotheses predicted "
    "elevation in the attention condition, so that session cannot separate them; "
    "it only rules out escape and tangible.\n\n"
    "Session 2 is an escape session with a low count. Automatic predicted elevation "
    "there and it did not occur, so automatic collapses. The tie is broken by a "
    "condition in which the two hypotheses disagree. This is the quantitative "
    "version of why an FA needs control and non-matching conditions: the "
    "discriminating evidence is in the conditions where responding is *absent*."
)

## Task 12: Plot the Posterior Trajectories for Both Datasets

Run the updating on both datasets and plot the posterior probability of each function against
session number, one panel per dataset. Use $\lambda_{\text{base}} = 2.0$ and
$\lambda_{\text{elevated}} = 2.6$ for the ambiguous dataset, again justified by its condition
means.

For each dataset report the first session at which the leading function exceeds 0.90 and the first
at which it exceeds 0.99.

Compare the two trajectories. Which one tells you something visual analysis could not?

In [ ]:
ambig_history = run_updating(ambig_df, base_rate=2.0, elevated_rate=2.6)

def first_crossing(history, function, threshold):
    hits = history.index[history[function] > threshold]
    return int(history.loc[hits[0], "session"]) if len(hits) else None

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, (name, hist, truth) in zip(
        axes,
        [("Clear (true function: attention)", clear_history, "attention"),
         ("Ambiguous (true function: escape)", ambig_history, "escape")]):
    for f in FUNCTIONS:
        ax.plot(hist.session, hist[f], marker="o", ms=3, label=f)
    ax.set_xlabel("Session")
    ax.set_ylim(0, 1)
    ax.set_title(name)
    ax.legend(title="Function", fontsize=8)
    p90, p99 = first_crossing(hist, truth, 0.90), first_crossing(hist, truth, 0.99)
    print(f"{name}: P({truth}) > 0.90 at session {p90}, > 0.99 at session {p99}")
axes[0].set_ylabel("Posterior probability")
plt.tight_layout()
plt.show()

print(
    "\nThe clear case resolves after two sessions and the plot is flat thereafter. "
    "Nothing in it that visual analysis did not already give you.\n\n"
    "The ambiguous case is where the method earns its place. The posterior climbs "
    "gradually, moves non-monotonically as individual sessions push back, and "
    "reaches high confidence only after several cycles. That trajectory is a "
    "running statement of how much the data support each function, including the "
    "cycles where the answer was still genuinely open. Visual analysis gives a "
    "verdict; it does not give a calibrated degree of belief, and it has no "
    "principled stopping rule."
)

## Task 13: Sensitivity to the Assumed Rates

$\lambda_{\text{base}}$ and $\lambda_{\text{elevated}}$ were assumptions, not estimates. A
conclusion that depends heavily on them is worth less than one that does not.

For each dataset, hold $\lambda_{\text{base}}$ fixed and sweep $\lambda_{\text{elevated}}$
across a range of plausible values. Record the final posterior probability of the true function for
each value, and plot or tabulate the result.

Which dataset's conclusion is robust to the assumption, and which is not? What does that imply
about reporting a Bayesian FA analysis?

In [ ]:
sweeps = {
    "Clear (attention)": (clear_df, 1.0, [5.0, 7.0, 9.0, 11.0, 13.0], "attention"),
    "Ambiguous (escape)": (ambig_df, 2.0, [2.2, 2.4, 2.6, 3.0, 3.5], "escape"),
}

for name, (df, base, grid, truth) in sweeps.items():
    print(f"--- {name}, base rate = {base} ---")
    for elev in grid:
        final = run_updating(df, base, elev).iloc[-1][truth]
        print(f"  elevated = {elev:4.1f}/min -> final P({truth}) = {final:.4f}")
    print()

print(
    "The clear dataset returns a final posterior of 1.0000 for attention at every "
    "assumed elevated rate from 5 to 13 per minute. The conclusion does not depend "
    "on getting the assumption right.\n\n"
    "The ambiguous dataset does depend on it: the final posterior for escape ranges "
    "from about 0.71 to 1.00 across a plausible range. The ordering is stable "
    "(escape stays on top), but the confidence is not.\n\n"
    "So a Bayesian FA analysis should report the assumed rates alongside the "
    "posterior, and should report a sensitivity sweep whenever the data are close. "
    "A posterior of 0.99 that becomes 0.71 under a slightly different assumption is "
    "not a 0.99."
)

---
# Part 3: Monte Carlo Simulation for Confidence Intervals

Not every quantity you care about has a tidy standard error. Simulation gives you an interval
regardless, by resampling the data you have.

## Task 14: Bootstrap the Attention-Condition Mean

Using the **clear** FA dataset, estimate the mean rate in the attention condition and put a 95%
confidence interval on it by bootstrap:

1. Extract the attention-condition rates.
2. Draw 10,000 resamples, each the same size as the original, sampled **with replacement**.
3. Compute the mean of each resample.
4. Take the 2.5th and 97.5th percentiles of those 10,000 means.
5. Plot the bootstrap distribution with the observed mean and both interval bounds marked.

Then compute the ordinary $t$-based interval on the same data and compare. Where do they differ,
and which assumption is each one making?

In [ ]:
attention_rates = clear_df.loc[clear_df.condition == "attention",
                               "rate_per_min"].values
n = len(attention_rates)
N_BOOT = 10_000

boot_means = np.array([
    rng.choice(attention_rates, size=n, replace=True).mean()
    for _ in range(N_BOOT)
])

ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
obs_mean = attention_rates.mean()
t_low, t_high = stats.t.interval(0.95, n - 1, loc=obs_mean,
                                 scale=stats.sem(attention_rates))

print(f"Observed mean:        {obs_mean:.2f} responses/min  (n = {n})")
print(f"95% bootstrap CI:     [{ci_low:.2f}, {ci_high:.2f}]  "
      f"(width {ci_high - ci_low:.2f})")
print(f"95% t-based CI:       [{t_low:.2f}, {t_high:.2f}]  "
      f"(width {t_high - t_low:.2f})")

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(boot_means, bins=40, color="steelblue", alpha=0.75)
ax.axvline(obs_mean, color="black", lw=2, label=f"observed mean = {obs_mean:.2f}")
ax.axvline(ci_low, color="firebrick", ls="--",
           label=f"2.5th pct = {ci_low:.2f}")
ax.axvline(ci_high, color="firebrick", ls="--",
           label=f"97.5th pct = {ci_high:.2f}")
ax.set_xlabel("Bootstrap mean rate, attention condition")
ax.set_ylabel("Frequency")
ax.set_title("Bootstrap Distribution of the Attention-Condition Mean")
ax.legend()
plt.tight_layout()
plt.show()

print(
    "\nThe two intervals are close, and the bootstrap is slightly narrower. The "
    "t interval assumes the sampling distribution of the mean is t-distributed, "
    "which follows if the underlying rates are roughly normal. The bootstrap "
    "assumes only that the sample is representative of the population, and it "
    "inherits whatever shape the data actually have. With n = 10 the bootstrap "
    "cannot represent tails that were never sampled, which is why it runs a little "
    "narrow here. Agreement between the two is evidence that the normal "
    "approximation is reasonable for these data, not evidence that either is right."
)

## Wrap-up

Answer the following in a markdown cell. These are the points the lab was built around.

1. Both subjects in Part 1 returned the same rate estimate. What is the general lesson about
   reporting a fitted parameter without reporting the checks that could have rejected the model?
2. In Part 2, the posterior after a single elevated attention session was split evenly between
   `attention` and `automatic`. What does that say about which sessions in an FA carry the
   discriminating evidence?
3. What are the advantages of Bayesian updating over visual analysis for FA interpretation, and
   what are its limitations? Be specific about what the prior and the likelihood are doing.
4. Under what circumstances would you trust the ambiguous dataset's final posterior of 0.9998, and
   under what circumstances would you not?
5. Where might a probabilistic approach to functional assessment be most valuable in practice?

In [ ]:
# Write your answers in a markdown cell below

### Wrap-up: Worked Answers

**1. Reporting a parameter without its diagnostics.** M-01 and M-02 returned rate estimates of
about 15 responses per minute with overlapping confidence intervals, and they are not behaving the
same way at all. The estimate answered the question it was asked -- what single rate best accounts
for these data -- and a maximum likelihood estimator will always answer that question, including
when the model is wrong. The confidence interval does not help, because it describes uncertainty
about $\lambda$ *given* that the model holds; it says nothing about whether it holds. Only the
predictions the model made and did not have fitted to it (variance equals mean, exponential IRTs)
can find the problem. A fitted parameter reported without those checks is a claim that the model
was never tested.

**2. Which sessions carry the discriminating evidence.** The elevated attention session confirmed
what both `attention` and `automatic` predicted, so it could not separate them; it only ruled out
the two hypotheses that predicted a low rate there. What broke the tie was the *escape* session, in
which responding was low: `automatic` had predicted elevation and was wrong. Evidence discriminates
between hypotheses only where they disagree. In an FA, the conditions where behavior is absent are
doing as much work as the condition where it is elevated, which is the formal reason a control
condition is not optional.

**3. Bayesian updating versus visual analysis.** The advantages are that the conclusion is a
calibrated distribution rather than a verdict, so partial evidence can be reported as partial
confidence; evidence accumulates automatically, with each posterior becoming the next prior, so
there is a principled basis for deciding whether another cycle is worth running; and every
assumption is written down. The prior states what was believed before the assessment, which can
encode base rates from the literature or from an interview. The likelihood is where the behavioral
theory lives: it is the statement that an attention function means elevated responding in the
attention condition and base-rate responding elsewhere, and that an automatic function means
elevation everywhere. The limitations follow from the same place. The posterior is only as good as
the likelihood, and the likelihood here is a deliberate simplification: it treats sessions as
independent, ignores within-session patterns and sequence effects, allows exactly one function, and
takes the base and elevated rates as known. It also cannot represent multiply-controlled behavior
at all, which is common. A confident posterior over a hypothesis space that omits the true function
is confidently wrong.

**4. When to trust the 0.9998.** Trust it to the extent that the model behind it survives scrutiny.
It is credible if the assumed rates are defensible and the sensitivity sweep in Task 13 shows the
conclusion is stable across the plausible range; if the four candidate functions plausibly exhaust
the possibilities for this client; and if sessions really are conditionally independent. The
sensitivity sweep is the reason for caution here: the same data yield about 0.71 under a slightly
lower assumed elevated rate. That the ordering held while the confidence moved by nearly 0.3 means
the *ranking* is well supported and the *number* is not. Report it as "escape is clearly the best
supported of the four, with confidence sensitive to the assumed rate," not as a probability of
0.9998. And any number computed here is conditional on a hypothesis space that excludes multiple
control.

**5. Where this is most valuable in practice.** Wherever visual analysis is weakest. Undifferentiated
or overlapping FA data, where analysts disagree and the honest answer is that the evidence is not
yet decisive. Situations where sessions are expensive or restricted, since a posterior gives a
defensible basis for deciding whether the next cycle would change the decision. Settings with
genuine prior information, such as a client with a documented history or a population with known
base rates, which visual analysis has no way to incorporate. And cases requiring an auditable
decision, where "the posterior probability of an escape function is 0.93 under these stated
assumptions" is a more inspectable claim than "the graph shows an escape function."